# Failure Mode: TCGA-05-4425 Darkspot / Foreign-Object Over-call

This notebook diagnoses the high class-3 burden observed for `TCGA-05-4425-01Z-00-DX1`. It measures the current inference path and does not edit model loading, slide reading, or label mapping.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from modules import grandqc_qc

SLIDE_ID = 'TCGA-05-4425-01Z-00-DX1'
ARTIFACT_MPP = 1.5
summary_path = REPO_ROOT / 'web' / 'output' / 'tcga_luad' / SLIDE_ID / 'summary.json'
tiles_path = REPO_ROOT / 'web' / 'output' / 'tcga_luad' / SLIDE_ID / 'tiles' / f'{SLIDE_ID}_tiles.parquet'
summary_path, tiles_path

## 1. Locate the highest class-3 tile

The dashboard stores per-tile parquet files for completed runs. This cell selects the highest darkspot/foreign-object tile from the existing direct-DICOM result.

In [ ]:
tiles = pd.read_parquet(tiles_path)
tiles['class3_fraction'] = tiles['darkspot_foreign_object_px'] / tiles['tissue_px'].replace(0, np.nan)
worst = tiles.sort_values('class3_fraction', ascending=False).iloc[0]
worst

## 2. Inspect MPP / scale

For a 0.5 um/px slide and a 1.5 um/px artifact model, the native patch is expected to be about 1536 px for a 512 px model tile.

In [ ]:
native_patch = int(worst['native_patch_size_px'])
effective_mpp = 0.5 * (native_patch / grandqc_qc.MODEL_TILE_SIZE)
print('native_patch_size_px:', native_patch)
print('model tile px:', grandqc_qc.MODEL_TILE_SIZE)
print('estimated effective artifact MPP:', effective_mpp)
print('target artifact MPP:', ARTIFACT_MPP)

## 3. Channel-order hypothesis

This re-runs artifact inference on the stored dashboard thumbnail crop if raw DICOM inputs are not present. For a fresh tile-level test, re-run the dashboard with raw retention enabled or use a local downloaded DICOM series and `grandqc_qc.open_slide`.

In [ ]:
thumb_path = REPO_ROOT / 'web' / 'output' / 'tcga_luad' / SLIDE_ID / 'tis_det_thumbnail' / f'{SLIDE_ID}.jpg'
thumb = Image.open(thumb_path).convert('RGB')
tile = thumb.resize((grandqc_qc.MODEL_TILE_SIZE, grandqc_qc.MODEL_TILE_SIZE), Image.Resampling.LANCZOS)
bgr_tile = Image.fromarray(np.asarray(tile)[..., ::-1])

device = grandqc_qc._default_device()
grandqc_qc.check_weights(artifact_mpp=ARTIFACT_MPP)
_, artifact_model, preprocessing_fn = grandqc_qc._load_models(ARTIFACT_MPP, device)

def predicted_counts(image):
    raw = grandqc_qc._predict_raw_artifact_mask(artifact_model, preprocessing_fn, image, device)
    documented = grandqc_qc._grandqc_raw_to_documented_labels(raw)
    return {int(k): int((documented == k).sum()) for k in np.unique(documented)}

pd.DataFrame([
    {'variant': 'rgb_thumbnail_proxy', 'counts': predicted_counts(tile)},
    {'variant': 'bgr_channel_swap_proxy', 'counts': predicted_counts(bgr_tile)},
])

## 4. Visual inspection

Use the overlay and class map exported by the dashboard to decide whether the class-3 region is plausible artifact or clean tissue mislabeled as darkspot/foreign object.

In [ ]:
base = REPO_ROOT / 'web' / 'output' / 'tcga_luad' / SLIDE_ID
paths = {
    'thumbnail': base / 'tis_det_thumbnail' / f'{SLIDE_ID}.jpg',
    'overlay': base / 'overlays_qc' / f'{SLIDE_ID}_overlay_QC.jpg',
    'map': base / 'maps_qc' / f'{SLIDE_ID}_map_QC.png',
}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, path) in zip(axes, paths.items()):
    ax.imshow(Image.open(path))
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()

## Conclusion template

Record the evidence here after running the cells:

- **Channel order:** supported / not supported, based on RGB vs BGR counts.
- **MPP scale:** supported / not supported, based on effective tile MPP.
- **Morphology:** plausible artifact / likely over-call, based on overlay and IDC SLIM review.
- **Recommendation:** If this is an inference bug, propose the one-line fix separately for approval. If it is model behavior on atypical morphology, keep the manual-review flag rule.